In [1]:
import json
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"
PRODUCT_DIR = PROJECT_ROOT / "data" / "processed" / "analytical_products"
INTEGRATED_PATH = PROJECT_ROOT / "data" / "processed" / "integrated_taxi_trips"

history = []
if RESULTS_PATH.exists():
    with RESULTS_PATH.open(encoding="utf-8") as handle:
        history = [json.loads(line) for line in handle if line.strip()]

print(f"Project root: {PROJECT_ROOT}")
print(f"Historical benchmark records: {len(history)}")

Project root: D:\Master\data intensive\data-intensive-computing
Historical benchmark records: 11


In [2]:
baseline_queries = pd.DataFrame([
    {"analysis": "Monthly demand by zone", "baseline_seconds": 101.717, "result_rows": 793, "controlled_optimization": "broadcast join; monthly pruning variant; AQE diagnostic"},
    {"analysis": "Distance by weather", "baseline_seconds": 22.686, "result_rows": 5, "controlled_optimization": "AQE diagnostic"},
    {"analysis": "Air-quality demand bands", "baseline_seconds": 11.322, "result_rows": 4, "controlled_optimization": "shared-intermediate caching"},
    {"analysis": "Air-quality demand correlation", "baseline_seconds": 8.578, "result_rows": 4, "controlled_optimization": "shared-intermediate caching"},
    {"analysis": "Zone weather-demand variation", "baseline_seconds": 23.920, "result_rows": 20, "controlled_optimization": "AQE diagnostic"},
    {"analysis": "Peak hours by weekday", "baseline_seconds": 15.166, "result_rows": 7, "controlled_optimization": "none; single full-history aggregation"},
    {"analysis": "Monthly demand trend", "baseline_seconds": 9.002, "result_rows": 8, "controlled_optimization": "none; single full-history aggregation"},
])
baseline_queries

,analysis,baseline_seconds,result_rows,controlled_optimization
0,Monthly demand by zone,101.717,793,broadcast join; monthly pruning variant; AQE d...
1,Distance by weather,22.686,5,AQE diagnostic
2,Air-quality demand bands,11.322,4,shared-intermediate caching
3,Air-quality demand correlation,8.578,4,shared-intermediate caching
4,Zone weather-demand variation,23.920,20,AQE diagnostic
5,Peak hours by weekday,15.166,7,none; single full-history aggregation
6,Monthly demand trend,9.002,8,none; single full-history aggregation


In [3]:
experiments = pd.DataFrame([
    {
        "query": "January 2024 zone demand variant",
        "technique": "partition pruning",
        "before_seconds": 0.2260,
        "after_seconds": 0.1801,
        "observed_ratio": 1.25,
        "validated_speedup": 1.25,
        "correctness_verified": True,
        "source_notebook": "optimization/partition_pruning.ipynb",
    },
    {
        "query": "monthly_zone_demand",
        "technique": "broadcast join",
        "before_seconds": 1.2314,
        "after_seconds": 0.3449,
        "observed_ratio": 3.57,
        "validated_speedup": 3.57,
        "correctness_verified": True,
        "source_notebook": "optimization/broadcast_join.ipynb",
    },
    {
        "query": "air-quality band + correlation pair",
        "technique": "caching",
        "before_seconds": 0.6080,
        "after_seconds": 0.1037,
        "observed_ratio": 5.86,
        "validated_speedup": 5.86,
        "correctness_verified": True,
        "source_notebook": "optimization/caching.ipynb",
    },
    {
        "query": "monthly_zone_demand",
        "technique": "AQE",
        "before_seconds": 0.4115,
        "after_seconds": 0.3922,
        "observed_ratio": 1.05,
        "validated_speedup": None,
        "correctness_verified": True,
        "source_notebook": "optimization/aqe_monthly_zone_demand.ipynb",
    },
    {
        "query": "distance_by_weather",
        "technique": "AQE",
        "before_seconds": 0.4041,
        "after_seconds": 0.2949,
        "observed_ratio": 1.37,
        "validated_speedup": None,
        "correctness_verified": True,
        "source_notebook": "optimization/aqe_distance_by_weather.ipynb",
    },
    {
        "query": "zone_weather_demand_variation",
        "technique": "AQE",
        "before_seconds": 0.9158,
        "after_seconds": 0.8001,
        "observed_ratio": 1.14,
        "validated_speedup": None,
        "correctness_verified": True,
        "source_notebook": "optimization/aqe_zone_weather_demand_variation.ipynb",
    },
])

assert experiments["correctness_verified"].all()
experiments

,query,technique,before_seconds,after_seconds,observed_ratio,validated_speedup,correctness_verified,source_notebook
0,January 2024 zone demand variant,partition pruning,0.2260,0.1801,1.25,1.25,True,optimization/partition_pruning.ipynb
1,monthly_zone_demand,broadcast join,1.2314,0.3449,3.57,3.57,True,optimization/broadcast_join.ipynb
2,air-quality band + correlation pair,caching,0.6080,0.1037,5.86,5.86,True,optimization/caching.ipynb
3,monthly_zone_demand,AQE,0.4115,0.3922,1.05,NaN,True,optimization/aqe_monthly_zone_demand.ipynb
4,distance_by_weather,AQE,0.4041,0.2949,1.37,NaN,True,optimization/aqe_distance_by_weather.ipynb
5,zone_weather_demand_variation,AQE,0.9158,0.8001,1.14,NaN,True,optimization/aqe_zone_weather_demand_variation...


In [4]:
optimization_coverage = pd.DataFrame([
    {"analysis": "Monthly demand by zone", "caching": "not reused", "partition_pruning": "1.25x on January slice", "broadcast_join": "3.57x", "AQE": "no effect"},
    {"analysis": "Distance by weather", "caching": "single use", "partition_pruning": "full history", "broadcast_join": "no join", "AQE": "no effect"},
    {"analysis": "Air-quality band + correlation", "caching": "5.86x", "partition_pruning": "full history", "broadcast_join": "already integrated", "AQE": "not selected"},
    {"analysis": "Zone weather-demand variation", "caching": "no reused intermediate", "partition_pruning": "full history", "broadcast_join": "not isolated", "AQE": "no effect"},
    {"analysis": "Peak hours by weekday", "caching": "single use", "partition_pruning": "full history", "broadcast_join": "no join", "AQE": "not selected"},
    {"analysis": "Monthly demand trend", "caching": "single use", "partition_pruning": "full history", "broadcast_join": "no join", "AQE": "not selected"},
])
optimization_coverage

,analysis,caching,partition_pruning,broadcast_join,AQE
0,Monthly demand by zone,not reused,1.25x on January slice,3.57x,no effect
1,Distance by weather,single use,full history,no join,no effect
2,Air-quality band + correlation,5.86x,full history,already integrated,not selected
3,Zone weather-demand variation,no reused intermediate,full history,not isolated,no effect
4,Peak hours by weekday,single use,full history,no join,not selected
5,Monthly demand trend,single use,full history,no join,not selected


In [5]:
plan_changes = pd.DataFrame([
    {
        "technique": "partition pruning",
        "before_plan": "PartitionFilters absent; row-level Filter after scan",
        "after_plan": "PartitionFilters contains pickup_year=2024 and pickup_month=1; row Filter removed",
        "interpretation": "Only the required Delta partitions are read",
    },
    {
        "technique": "broadcast join",
        "before_plan": "ShuffledHashJoin with Exchange on both inputs",
        "after_plan": "BroadcastHashJoin and BroadcastExchange for the 265-row taxi_zones table",
        "interpretation": "The multi-million-row taxi fact table is not shuffled",
    },
    {
        "technique": "caching",
        "before_plan": "Each query repeats source Scan, Filter, and HashAggregate",
        "after_plan": "Both downstream queries use InMemoryTableScan",
        "interpretation": "The 6,435-row borough-hour aggregate is computed once and reused",
    },
    {
        "technique": "AQE (all three queries)",
        "before_plan": "Static physical plan",
        "after_plan": "AdaptiveSparkPlan wrapper only; no AQEShuffleRead; Final Plan equals Initial Plan",
        "interpretation": "AQE made no runtime adjustment",
    },
])
plan_changes

,technique,before_plan,after_plan,interpretation
0,partition pruning,PartitionFilters absent; row-level Filter afte...,PartitionFilters contains pickup_year=2024 and...,Only the required Delta partitions are read
1,broadcast join,ShuffledHashJoin with Exchange on both inputs,BroadcastHashJoin and BroadcastExchange for th...,The multi-million-row taxi fact table is not s...
2,caching,"Each query repeats source Scan, Filter, and Ha...",Both downstream queries use InMemoryTableScan,"The 6,435-row borough-hour aggregate is comput..."
3,AQE (all three queries),Static physical plan,AdaptiveSparkPlan wrapper only; no AQEShuffleR...,AQE made no runtime adjustment


In [6]:
aqe_order_checks = pd.DataFrame([
    {
        "query": "distance_by_weather",
        "original_first": "AQE off: 0.4041s",
        "original_second": "AQE on: 0.2949s",
        "reverse_first": "AQE on: 0.4004s",
        "reverse_second": "AQE off: 0.2756s",
        "conclusion": "winner followed second position; no AQE effect",
    },
    {
        "query": "monthly_zone_demand",
        "original_first": "AQE off: 0.4115s",
        "original_second": "AQE on: 0.3922s",
        "reverse_first": "AQE on: 0.4687s",
        "reverse_second": "AQE off: 0.3997s",
        "conclusion": "winner followed second position; no AQE effect",
    },
    {
        "query": "zone_weather_demand_variation",
        "original_first": "AQE off: 0.9158s",
        "original_second": "AQE on: 0.8001s",
        "reverse_first": "not run",
        "reverse_second": "not run",
        "conclusion": "no plan adaptation; 1.14x is not accepted as causal",
    },
])
aqe_order_checks

,query,original_first,original_second,reverse_first,reverse_second,conclusion
0,distance_by_weather,AQE off: 0.4041s,AQE on: 0.2949s,AQE on: 0.4004s,AQE off: 0.2756s,winner followed second position; no AQE effect
1,monthly_zone_demand,AQE off: 0.4115s,AQE on: 0.3922s,AQE on: 0.4687s,AQE off: 0.3997s,winner followed second position; no AQE effect
2,zone_weather_demand_variation,AQE off: 0.9158s,AQE on: 0.8001s,not run,not run,no plan adaptation; 1.14x is not accepted as c...


In [7]:
from data_intensive_computing.spark_env import configure_windows_spark_env

configure_windows_spark_env()

from delta import DeltaTable, configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("Lab2-Platform-Evaluation")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

PRODUCT_NAMES = [
    "zone_hourly_demand_profile",
    "airport_transfer_summary",
    "weather_sensitive_zone_demand",
    "origin_destination_flow",
    "fare_and_tip_patterns",
    "trip_distance_distribution",
]
REQUIRED_METADATA = {"source_table", "created_at", "refreshed_at", "schema_version"}


def delta_metrics(path: Path):
    detail = DeltaTable.forPath(spark, str(path)).detail().first()
    frame = spark.read.format("delta").load(str(path))
    return {
        "rows": frame.count(),
        "files": int(detail["numFiles"]),
        "size_bytes": int(detail["sizeInBytes"]),
        "metadata_complete": REQUIRED_METADATA.issubset(frame.columns),
    }


if not (INTEGRATED_PATH / "_delta_log").exists():
    raise FileNotFoundError(f"Missing integrated Delta table: {INTEGRATED_PATH}")

source_detail = DeltaTable.forPath(spark, str(INTEGRATED_PATH)).detail().first()
source_size_bytes = int(source_detail["sizeInBytes"])
storage_rows = []
missing_products = []

for product_name in PRODUCT_NAMES:
    path = PRODUCT_DIR / product_name
    if not (path / "_delta_log").exists():
        missing_products.append(product_name)
        continue
    metrics = delta_metrics(path)
    storage_rows.append({
        "product": product_name,
        "rows": metrics["rows"],
        "active_data_files": metrics["files"],
        "storage_mb": round(metrics["size_bytes"] / 1024**2, 3),
        "pct_of_integrated_table": round(100 * metrics["size_bytes"] / source_size_bytes, 3),
        "metadata_complete": metrics["metadata_complete"],
    })

product_storage = pd.DataFrame(storage_rows)
if missing_products:
    print("Missing products; run their notebooks before final reporting:", missing_products)
if not product_storage.empty:
    display(product_storage)
    print(f"Integrated table size: {source_size_bytes / 1024**2:.3f} MB")
    print(f"Total product storage: {product_storage['storage_mb'].sum():.3f} MB")
    print(
        "Total storage overhead relative to integrated table: "
        f"{product_storage['pct_of_integrated_table'].sum():.3f}%"
    )
    assert product_storage["metadata_complete"].all()
else:
    print("No materialized analytical products are currently available to measure.")

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,product,rows,active_data_files,storage_mb,pct_of_integrated_table,metadata_complete
0,zone_hourly_demand_profile,240749,40,3.599,0.384,True
1,airport_transfer_summary,2847,1,0.065,0.007,True
2,weather_sensitive_zone_demand,3930,1,0.065,0.007,True
3,origin_destination_flow,82847,40,2.053,0.219,True
4,fare_and_tip_patterns,131,1,0.010,0.001,True
5,trip_distance_distribution,133,1,0.008,0.001,True


Integrated table size: 936.188 MB
Total product storage: 5.800 MB
Total storage overhead relative to integrated table: 0.619%


In [8]:
# Confirm that the consolidated controlled measurements exist in the JSONL history.
expected_history_records = [
    ("zone_demand_jan2024_single_table", "partition_pruning", 0.2260, 0.1801),
    ("monthly_zone_demand", "broadcast_join", 1.2314, 0.3449),
    ("air_quality_demand_by_band_and_correlation", "caching", 0.6080, 0.1037),
    ("monthly_zone_demand", "aqe", 0.4115, 0.3922),
    ("distance_by_weather", "aqe", 0.4041, 0.2949),
    ("zone_weather_demand_variation", "aqe", 0.9158, 0.8001),
]

for query_name, technique, before, after in expected_history_records:
    matched = any(
        row.get("query_name") == query_name
        and row.get("technique") == technique
        and abs(row.get("before_time_sec", -1) - before) < 1e-9
        and abs(row.get("after_time_sec", -1) - after) < 1e-9
        for row in history
    )
    assert matched, f"Missing expected benchmark record: {query_name} / {technique}"

print("All consolidated timing records were found in the benchmark history.")

All consolidated timing records were found in the benchmark history.
